<a href="https://colab.research.google.com/github/JustinRSK/2025_ML_EES/blob/main/Project/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip -q install rasterio scikit-learn numpy matplotlib


In [7]:
import os
print("Features file size (MB):", os.path.getsize("/content/Features_2056_clean.tif")/1e6)


Features file size (MB): 165.283696


In [8]:
import numpy as np
import rasterio

paths = {
    "B2":  "/content/B2_2056.tif",
    "B3":  "/content/B3_2056.tif",
    "B4":  "/content/B4_2056.tif",
    "NDVI":"/content/NDVI_2056.tif",
    "NDWI":"/content/NDWI_2056.tif",
}

arrays = []
ref_profile = None

for k in ["B2","B3","B4","NDVI","NDWI"]:
    with rasterio.open(paths[k]) as src:
        if ref_profile is None:
            ref_profile = src.profile.copy()
            ref_shape = (src.height, src.width)
            ref_transform = src.transform
            ref_crs = src.crs
        else:
            assert (src.height, src.width) == ref_shape, f"Grid mismatch for {k}"
            assert src.transform == ref_transform, f"Transform mismatch for {k}"
            assert src.crs == ref_crs, f"CRS mismatch for {k}"

        arrays.append(src.read(1).astype(np.float32))

stack = np.stack(arrays, axis=0)  # (5, H, W)

out_profile = ref_profile.copy()
out_profile.update(count=5, dtype=rasterio.float32)

out_path = "/content/Features_2056_FIXED.tif"
with rasterio.open(out_path, "w", **out_profile) as dst:
    dst.write(stack)

print("Wrote:", out_path, "shape:", stack.shape)


Wrote: /content/Features_2056_FIXED.tif shape: (5, 3319, 4251)


In [9]:
import rasterio
with rasterio.open("/content/Features_2056_FIXED.tif") as src:
    print("bands:", src.count, "size:", src.width, src.height, "dtype:", src.dtypes)


bands: 5 size: 4251 3319 dtype: ('float32', 'float32', 'float32', 'float32', 'float32')


In [10]:
features_path = "/content/Features_2056_FIXED.tif"
labels_path   = "/content/labels_5classes_2056.tif"


In [12]:
import rasterio

features_path = "/content/Features_2056_FIXED.tif"
labels_path   = "/content/labels_5classes_2056.tif"

with rasterio.open(features_path) as fx:
    print("FEATURES")
    print(" shape:", (fx.count, fx.height, fx.width))
    print(" crs:", fx.crs)
    print(" transform:", fx.transform)
    print(" bounds:", fx.bounds)

with rasterio.open(labels_path) as ly:
    print("\nLABELS")
    print(" shape:", (ly.count, ly.height, ly.width))
    print(" crs:", ly.crs)
    print(" transform:", ly.transform)
    print(" bounds:", ly.bounds)


FEATURES
 shape: (5, 3319, 4251)
 crs: EPSG:2056
 transform: | 7.65, 0.00, 2538302.40|
| 0.00,-7.65, 1206408.54|
| 0.00, 0.00, 1.00|
 bounds: BoundingBox(left=2538302.4014, bottom=1181007.8291, right=2570835.8277, top=1206408.5447)

LABELS
 shape: (1, 2540, 3253)
 crs: EPSG:2056
 transform: | 10.00, 0.00, 2538302.40|
| 0.00,-10.00, 1206408.54|
| 0.00, 0.00, 1.00|
 bounds: BoundingBox(left=2538302.4014, bottom=1181008.5447, right=2570832.4014, top=1206408.5447)


In [13]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

features_path = "/content/Features_2056_FIXED.tif"
labels_path   = "/content/labels_5classes_2056.tif"
labels_aligned_path = "/content/labels_5classes_2056_ALIGNED.tif"

with rasterio.open(features_path) as fx:
    dst_crs = fx.crs
    dst_transform = fx.transform
    dst_height = fx.height
    dst_width = fx.width
    fx_profile = fx.profile.copy()

with rasterio.open(labels_path) as src:
    src_data = src.read(1)
    src_transform = src.transform
    src_crs = src.crs
    src_nodata = src.nodata if src.nodata is not None else 0

# destination array
dst_data = np.zeros((dst_height, dst_width), dtype=np.uint8)

reproject(
    source=src_data,
    destination=dst_data,
    src_transform=src_transform,
    src_crs=src_crs,
    dst_transform=dst_transform,
    dst_crs=dst_crs,
    resampling=Resampling.nearest,   # IMPORTANT for class labels
    src_nodata=src_nodata,
    dst_nodata=0
)

# save aligned labels
out_profile = fx_profile.copy()
out_profile.update(count=1, dtype=rasterio.uint8, nodata=0)

with rasterio.open(labels_aligned_path, "w", **out_profile) as dst:
    dst.write(dst_data, 1)

print("Saved:", labels_aligned_path)
print("Unique label values:", np.unique(dst_data))


Saved: /content/labels_5classes_2056_ALIGNED.tif
Unique label values: [0 1 2 3 4 5]


In [14]:
labels_path = "/content/labels_5classes_2056_ALIGNED.tif"


In [15]:
import numpy as np
import rasterio

features_path = "/content/Features_2056_FIXED.tif"
labels_path   = "/content/labels_5classes_2056_ALIGNED.tif"

with rasterio.open(features_path) as srcX:
    X = srcX.read()  # (bands, rows, cols)
    profileX = srcX.profile

with rasterio.open(labels_path) as srcY:
    y = srcY.read(1)  # (rows, cols)

print("Features:", X.shape, "Labels:", y.shape)
assert X.shape[1:] == y.shape

X_img = np.moveaxis(X, 0, -1)  # (rows, cols, bands)
mask = (y > 0)

X_all = X_img[mask]
y_all = y[mask].astype(int)

print("Training samples:", X_all.shape[0], "Bands:", X_all.shape[1])
print("Class counts (1..5):", np.bincount(y_all)[1:])


Features: (5, 3319, 4251) Labels: (3319, 4251)
Training samples: 5644 Bands: 5
Class counts (1..5): [ 245  662  395 1255 3087]


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

def tile_ids(rows, cols, tile=128):
    rr = np.arange(rows)[:, None] // tile
    cc = np.arange(cols)[None, :] // tile
    return rr * (cc.max() + 1) + cc

rows, cols = y.shape
tiles = tile_ids(rows, cols, tile=128)

unique_tiles = np.unique(tiles[mask])
rng = np.random.default_rng(42)
n_test = max(1, int(0.2 * len(unique_tiles)))
test_tiles = rng.choice(unique_tiles, size=n_test, replace=False)

train_mask = mask & (~np.isin(tiles, test_tiles))
test_mask  = mask & ( np.isin(tiles, test_tiles))

X_train = X_img[train_mask]
y_train = y[train_mask].astype(int)
X_test  = X_img[test_mask]
y_test  = y[test_mask].astype(int)

print("Train:", X_train.shape, "Test:", X_test.shape)

rf = RandomForestClassifier(
    n_estimators=400,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=42
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred, digits=3))


Train: (4606, 5) Test: (1038, 5)
Confusion matrix:
 [[ 50   4   2   7   0]
 [ 15 120   1   2   0]
 [  1   1   9  90   0]
 [  5   4  11  76   0]
 [  0   0   0   0 640]]

Report:
               precision    recall  f1-score   support

           1      0.704     0.794     0.746        63
           2      0.930     0.870     0.899       138
           3      0.391     0.089     0.145       101
           4      0.434     0.792     0.561        96
           5      1.000     1.000     1.000       640

    accuracy                          0.862      1038
   macro avg      0.692     0.709     0.670      1038
weighted avg      0.861     0.862     0.847      1038



Remap labels (merge 3+4, and move lake to 4)

In [18]:
import numpy as np
import rasterio

features_path = "/content/Features_2056_FIXED.tif"
labels_path   = "/content/labels_5classes_2056_ALIGNED.tif"

with rasterio.open(features_path) as fx:
    X = fx.read()
    profileX = fx.profile

with rasterio.open(labels_path) as ly:
    y = ly.read(1)

# y_new classes: 1,2,3(forest),4(lake), 0 background
y_new = y.copy()

# merge forests
y_new[(y == 3) | (y == 4)] = 3

# move lake from 5 -> 4
y_new[y == 5] = 4

print("Unique values (should be 0,1,2,3,4):", np.unique(y_new))


Unique values (should be 0,1,2,3,4): [0 1 2 3 4]


Train RF on the 4 clean classes

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

X_img = np.moveaxis(X, 0, -1)  # (rows, cols, bands)
mask = (y_new > 0)

# spatial split by tiles (same as before)
def tile_ids(rows, cols, tile=128):
    rr = np.arange(rows)[:, None] // tile
    cc = np.arange(cols)[None, :] // tile
    return rr * (cc.max() + 1) + cc

rows, cols = y_new.shape
tiles = tile_ids(rows, cols, tile=128)
unique_tiles = np.unique(tiles[mask])

rng = np.random.default_rng(42)
n_test = max(1, int(0.2 * len(unique_tiles)))
test_tiles = rng.choice(unique_tiles, size=n_test, replace=False)

train_mask = mask & (~np.isin(tiles, test_tiles))
test_mask  = mask & ( np.isin(tiles, test_tiles))

X_train = X_img[train_mask]
y_train = y_new[train_mask].astype(int)
X_test  = X_img[test_mask]
y_test  = y_new[test_mask].astype(int)

rf = RandomForestClassifier(
    n_estimators=400,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=42
)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print("\nReport:\n", classification_report(y_test, pred, digits=3))


Confusion matrix:
 [[ 49   4  10   0]
 [  8 119  11   0]
 [  1   0 196   0]
 [  0   0   0 640]]

Report:
               precision    recall  f1-score   support

           1      0.845     0.778     0.810        63
           2      0.967     0.862     0.912       138
           3      0.903     0.995     0.947       197
           4      1.000     1.000     1.000       640

    accuracy                          0.967      1038
   macro avg      0.929     0.909     0.917      1038
weighted avg      0.968     0.967     0.967      1038



Create class 5 = “Other/Unknown” after prediction

In [20]:
proba = rf.predict_proba(X_img.reshape(-1, X_img.shape[-1]))  # probs for classes 1..4
pred4 = rf.classes_[np.argmax(proba, axis=1)]                 # predicted class 1..4
conf  = np.max(proba, axis=1)

threshold = 0.60  # try 0.55–0.75
pred5 = pred4.copy()
pred5[conf < threshold] = 5  # low confidence => Unknown/Other

pred_map = pred5.reshape(rows, cols).astype(np.uint8)

print("Unknown share:", (pred_map==5).mean())


Unknown share: 0.018250460040984986


In [21]:
out_path = "/content/prediction_5cats_forest_unknown.tif"
out_profile = profileX.copy()
out_profile.update(count=1, dtype=rasterio.uint8, nodata=0)

with rasterio.open(out_path, "w", **out_profile) as dst:
    dst.write(pred_map, 1)

print("Saved:", out_path)


Saved: /content/prediction_5cats_forest_unknown.tif


Option 2 : Keep 4 categories

In [22]:
# Predict probabilities
proba = rf.predict_proba(X_img.reshape(-1, X_img.shape[-1]))

# Predicted class (1–4)
pred = rf.classes_[np.argmax(proba, axis=1)]

# Reshape to map
pred_map = pred.reshape(rows, cols).astype(np.uint8)


In [23]:
# Confidence = max probability per pixel
conf_map = np.max(proba, axis=1).reshape(rows, cols).astype(np.float32)


In [25]:
import rasterio
import numpy as np

# Ensure pred_map is uint8 and has a valid nodata (0)
pred_map_u8 = pred_map.astype(np.uint8)

# If you have any NaNs (unlikely for pred_map), convert to 0
pred_map_u8 = np.where(np.isfinite(pred_map_u8), pred_map_u8, 0).astype(np.uint8)

# Profiles
profile_cls = fx.profile.copy()
profile_cls.update(
    dtype=rasterio.uint8,
    count=1,
    nodata=0  # valid for uint8
)

profile_conf = fx.profile.copy()
profile_conf.update(
    dtype=rasterio.float32,
    count=1
)
# For confidence, it's fine to keep float nodata, but simplest:
profile_conf.pop("nodata", None)

# Save classification (4 classes)
with rasterio.open("/content/prediction_4classes.tif", "w", **profile_cls) as dst:
    dst.write(pred_map_u8, 1)

# Save confidence
with rasterio.open("/content/prediction_confidence.tif", "w", **profile_conf) as dst:
    dst.write(conf_map.astype(np.float32), 1)

print("Saved: prediction_4classes.tif and prediction_confidence.tif")


Saved: prediction_4classes.tif and prediction_confidence.tif


In [ ]:
Part 3 : Improving the model

label edge cleaning

In [26]:
# OPTIONAL: remove boundary pixels (edge noise) using a simple erosion-like rule:
# We'll drop pixels that have a different label in their 3x3 neighborhood.
import numpy as np
from scipy.ndimage import generic_filter

def keep_only_homogeneous(pixel_block):
    center = pixel_block[len(pixel_block)//2]
    if center == 0:
        return 0
    # if any neighbor differs from center (and is non-zero), drop it
    if np.any((pixel_block != center) & (pixel_block != 0)):
        return 0
    return center

y_clean = generic_filter(y, keep_only_homogeneous, size=3, mode="nearest").astype(y.dtype)

print("Before:", np.unique(y), "After:", np.unique(y_clean))
y = y_clean


Before: [0 1 2 3 4 5] After: [0 1 2 3 4 5]


Retrain a better Random Forest

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Build training dataset from pixels where y != 0 (0 = NoData/background)
mask = (y != 0)
X_samples = X_img[mask]     # (n_samples, n_features)
y_samples = y[mask]         # (n_samples,)

X_train, X_test, y_train, y_test = train_test_split(
    X_samples, y_samples,
    test_size=0.2,
    random_state=42,
    stratify=y_samples
)

rf = RandomForestClassifier(
    n_estimators=700,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    max_features="sqrt",
    min_samples_leaf=3,
    min_samples_split=10
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))


Confusion matrix:
 [[ 42   3   1   3   0]
 [  0 128   2   2   0]
 [  1   2  50  26   0]
 [  4   3  21 223   0]
 [  0   0   0   0 618]]

Report:
               precision    recall  f1-score   support

           1       0.89      0.86      0.88        49
           2       0.94      0.97      0.96       132
           3       0.68      0.63      0.65        79
           4       0.88      0.89      0.88       251
           5       1.00      1.00      1.00       618

    accuracy                           0.94      1129
   macro avg       0.88      0.87      0.87      1129
weighted avg       0.94      0.94      0.94      1129



In [28]:
proba = rf.predict_proba(X_img.reshape(-1, X_img.shape[-1]))
pred = rf.classes_[np.argmax(proba, axis=1)]
pred_map = pred.reshape(rows, cols).astype(np.uint8)

conf_map = np.max(proba, axis=1).reshape(rows, cols).astype(np.float32)


In [29]:
from scipy.ndimage import median_filter

pred_smooth = median_filter(pred_map, size=3)  # simple smoothing
pred_map = pred_smooth.astype(np.uint8)
